# Mandatory Assignment 2: Ensemble Models & Artificial Neural Networks
#### Group: 04
Group members: Brage Blomkvist, Patrick Rafael Jemieljanczyk & William Bjørling Nilsen

In [ ]:
# Import required libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cosine
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, TargetEncoder, FunctionTransformer
from sklearn.preprocessing import TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier


from sklearn.tree import DecisionTreeClassifier


## Description of Dataset

The dataset used is a set of daily weather observations from Australia. The observations are taken from a span of ten years. Various different locations are used

The dataset can be found here: https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package/data

### Problem

In [ ]:
# Read the dataset
dataset_AUS = pd.read_csv('Dataset/weatherAUS.csv')
# Uses existing column names baked into csv file.

# Show the first 5 rows to validate that the data import has been successful
dataset_AUS.head()



## Data Processing

In [ ]:
# Overview of dataset
# Look at the implementation details, data types and missing data
dataset_AUS.info()

In [ ]:
# See how many different locations are used.
dataset_AUS.value_counts("Location").nunique()


In [ ]:
# Check if every location has the same amount of observations
dataset_AUS.value_counts("Location")

### Missing Data

In [ ]:
# Look at the amount of missing values
dataset_AUS.isna().sum()

In [ ]:
dataset_AUS.count()

### Splitting the Dataset
Created temporary splitting of data


In [ ]:
# Gridsearch can be used, but Nested Cross validation should be considered first. Must use Cross validation for evaluation instead of random states.

# Creating a temporary dataset
dataset_AUS_temp = pd.read_csv('Dataset/weatherAUS.csv')

# Temporary separating into targets and features
y = dataset_AUS_temp['RainTomorrow']
x = dataset_AUS_temp.drop(columns=['RainTomorrow'])

# Temporary splitting
# Split into training and testing
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
)

# Select numerical columns
numerical_features = x.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

# Select categorical columns
categorical_features = x.select_dtypes(
    include=['object', 'category', 'string']
).columns.tolist()

# Pipelines
to do:
   - Make the processor able to remove certain columns. the one who runs the program can then choose between removing the columns or not, to see the differences this creates in the models.

   - Can make it an option to delete the feature day_of_year

   - Make an option to delete temp3am and temp9am


In [ ]:
# Putting the features which need to be encoded into variables as a list. To be used by pipelines to encode
location_features = ['Location']

rain_today_features = ['RainToday']

wind_features = [
    'WindGustDir',
    'WindDir9am',
    'WindDir3pm'
]

date_features = ['Date']

In [ ]:
# Cyclical encoder function
def cyclic_encoder(series, period):
    sin = np.sin(2 * np.pi * series / period)
    cos = np.cos(2 * np.pi * series / period)

    return sin, cos


In [ ]:
wind_map = {
    'N': 1, 'NNE': 2, 'NE': 3, 'ENE': 4,
    'E': 5, 'ESE': 6, 'SE': 7, 'SSE': 8,
    'S': 9, 'SSW': 10, 'SW': 11, 'WSW': 12,
    'W': 13, 'WNW': 14, 'NW': 15, 'NNW': 16
}

WIND_DIRECTIONS = 16

In [ ]:
def encode_wind(wind_data_frame):
    result = pd.DataFrame()

    for column in wind_data_frame.columns:
        # Converting the compass directions to numbers as seen in the wind_map
        wind_numbers = wind_data_frame[column].map(wind_map).astype(float)

        # Replacing the numbers with corresponding sine and cosine values for the circular data
        sin, cos = cyclic_encoder(wind_numbers, WIND_DIRECTIONS)
        result[column + '_Sine'], result[column + '_Cosine'] = sin, cos

    return result

In [ ]:
# Numerical pipeline
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Location pipeline
location_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', TargetEncoder(
        target_type='binary',
    ))
])

# Wind pipeline
wind_cyclical_pipeline = Pipeline([
    ('encoder', FunctionTransformer(
        encode_wind,
    )),
    ('imputer', SimpleImputer(
        strategy='constant',
    ))
])




In [ ]:
# Preprocessor
preprocessor = ColumnTransformer([
    ('numerical', numerical_pipeline, numerical_features),

    ('location', location_pipeline, location_features),

    ('rain_today', rain_today_pipeline, rain_today_features),

    ('wind', wind_cyclical_pipeline, wind_features),

    ('date', date_cyclical_pipeline, date_features)
])

In [ ]:
# Feature encoders
# Location encoder
location_target_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', TargetEncoder(
        #target_type='binary',
    ))
])


numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', TargetEncoder())
])

# Decision tree pipeline
dt_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', DecisionTreeClassifier())
])

# Logistic Regression pipeline
lr_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', LogisticRegression(max_iter=500))
])

# K-nearest neighbours
knn_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', KNeighborsClassifier(
        n_neighbors=5
    ))
])

# train decision tree
#dt_pipe.fit(x_train, y_train)

### Normalization

### Encoding

## Baseline Models

In [ ]:
# (Automated tuning)

## Advanced Models

## Advanced Neural Networks

## Evaluation & Comparison

### Testing

#### Combined Results Table

### Confusion Matrices & Additional Plots

### Conclusion